In [ ]:
import os
import time
import random
import traceback
import pandas as pd
from tqdm import tqdm
import concurrent.futures
from urllib.parse import urlparse
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

In [ ]:
file_path = 'C://Users//sohyun//PROJECT//Crawler//Artist_num_gallery.csv'
df = pd.read_csv(file_path)
urls = df['Main url'].tolist()
artists = df['Artist'].tolist()

In [ ]:
# WebDriver 설정
def init_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--ignore-certificate-errors')
    options.add_argument('--ignore-ssl-errors')
    return webdriver.Chrome(options=options)

driver = init_driver()
wait = WebDriverWait(driver, 10)

In [ ]:
# 크롤링할 링크 설정
num = 1

try:
    driver.get(urls[num])
    print(f"크롤링 대상: {artists[num]}\n{urls[num]}")

    # 게시글 링크 수집
    post_links = []

    # 끝 페이지의 번호 추출
    try:
        last_page_element = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a.page_end'))
        )
        last_page_url = last_page_element.get_attribute('href')
        last_page_number = int(last_page_url.split('=')[-1])
    except TimeoutException:
        print("Timeout occurred: 'page_end' element not found.")
        last_page_number = 1

    # 페이지 순회하며 게시글 링크 추출
    for i in tqdm(range(last_page_number)):
        page_url = urls[num][:-1] + str(i + 1)

        try:
            driver.get(page_url)
        except (NoSuchWindowException, WebDriverException):
            print("Driver issue detected. Reinitializing driver...")
            driver.quit()
            driver = init_driver()
            driver.get(page_url)

        # 게시글 링크 추출
        try:
            post_elements = driver.find_elements(By.CSS_SELECTOR, 'tbody.listwrap2 tr.us-post')
            for post in post_elements:
                try:
                    link_element = post.find_element(By.CSS_SELECTOR, 'td.gall_tit a[view-msg]')
                    href = link_element.get_attribute('href')
                    post_links.append(href)
                except NoSuchElementException:
                    continue
        except Exception as e:
            print(f"Error while extracting links on page {i + 1}: {e}")

except Exception as e:
    print(f"Unexpected error occurred: {e}")

len(post_links)

크롤링 대상: SHINee (샤이니)
https://gall.dcinside.com/board/lists/?id=shinee_new&page=1


  0%|                                                                            | 7/141045 [00:09<55:08:27,  1.41s/it]

Unexpected error occurred: name 'NoSuchWindowException' is not defined


349

In [ ]:
# 게시글 크롤링 함수
def crawling(post_link):

    try:
        driver.get(post_link)
        time.sleep(random.uniform(2, 3))

        try:
            WebDriverWait(driver, 2).until(
                EC.presence_of_element_located((By.CLASS_NAME, 'write_div'))
            )
        except TimeoutException:
            print(f"Timeout occurred for {post_link}: 'write_div' element not found.")
            return None

        # 게시글 기본 정보, 내용, 이미지, 댓글 수집
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
            future_basic_info = executor.submit(lambda: {
                'content_writer': driver.find_element(By.CLASS_NAME, 'nickname').text,
                'content_num': driver.find_element(By.CLASS_NAME, 'btn_recom_up').get_attribute("data-no"),
                'title': driver.find_element(By.CLASS_NAME, 'title_subject').text.strip(),
                'date': driver.find_element(By.CLASS_NAME, 'gall_date').text.strip(),
                'rec': driver.find_element(By.CLASS_NAME, 'gall_reply_num').text.strip().split(' ')[-1],
                'rec_down': driver.find_element(By.CLASS_NAME, 'down_num').text.strip(),
                'view': driver.find_element(By.CLASS_NAME, 'gall_count').text.strip().split(' ')[-1],
            })
            future_content = executor.submit(lambda: driver.find_element(By.CLASS_NAME, 'write_div').text.strip())
            future_images = executor.submit(lambda: "<image>" if driver.find_elements(By.CLASS_NAME, 'write_div img') else "")
            future_comments = executor.submit(lambda: [
                {
                    'comment': li.find_element(By.CLASS_NAME, 'cmt_txtbox').text.strip(),
                    'comment_date': li.find_element(By.CLASS_NAME, 'date_time').text.strip(),
                    'comment_writer': (li.find_element(By.CLASS_NAME, 'nickname').text if li.find_elements(By.CLASS_NAME, 'nickname') else None),
                    'comment_ip': (li.find_element(By.CLASS_NAME, 'ip').get_attribute("ip") if li.find_elements(By.CLASS_NAME, 'ip') else None),
                    'comment_no': li.find_element(By.CSS_SELECTOR, '[data-no]').text.strip()
                }
                for li in driver.find_elements(By.CLASS_NAME, 'cmt_list li') if not li.find_elements(By.CLASS_NAME, 'del_reply')
            ])

            basic_info = future_basic_info.result()
            content = future_content.result()
            content_imgs = future_images.result()
            comments = future_comments.result()

        getall.append({
            'URL': post_link,
            '작성자': basic_info['content_writer'],
            '게시글번호': basic_info['content_num'],
            '제목': basic_info['title'],
            '작성날짜': basic_info['date'],
            '추천': basic_info['rec'],
            '비추천': basic_info['rec_down'],
            '조회수': basic_info['view'],
            '게시글': content,
            '이미지': content_imgs,
            '댓글': comments
        })
        print("****크롤링 완료****\n")

    except Exception as e:
        print(f"Error while processing {post_link}: {e}")
        print(traceback.format_exc())
        error_link.append(post_link)

In [ ]:
# 크롤링 실행
# 끊겼을 경우 getall, error_link 주석처리 후, 크롤링 끊긴 번호로 i 수정하여 다시 크롤링 진행

getall = [] # 끊겼을 경우 주석처리!
error_link = [] # 끊겼을 경우 주석처리!

i=0 # 끊겼을 경우 크롤링 완료된 다음 번호로 수정
for n in range(len(post_links)):
    driver.get(post_links[i])
    time.sleep(1)
    print(f"[{i}/{len(post_links)}]")
    crawling(post_links[i])
    i+=1

[0/349]
****크롤링 완료****

[1/349]
****크롤링 완료****

[2/349]
****크롤링 완료****

[3/349]
****크롤링 완료****



KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(getall)
pd.set_option('display.max_rows', None)
df.to_csv(f'{artists[num]}.csv', index=False, encoding='utf-8-sig')
print(f"{artists[num]} 저장 완료")

SHINee (샤이니) 저장 완료


In [ ]:
error_link